# 🧠 ALETHEIA FINE-TUNE (ONE CLICK)

## Instructions:
1. Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all** (Ctrl+F9)
3. When prompted, **upload** your `aletheia_train_XXXX.jsonl.gz` file
4. Wait ~45 minutes
5. Download the model files when prompted

That's it! No coding knowledge needed.

In [ ]:
#@title **Step 1: Choose Your Model** { display-mode: "form" }
#@markdown Select which base model to fine-tune:
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" #@param ["unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", "unsloth/Llama-3.2-1B-Instruct-bnb-4bit", "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", "unsloth/Phi-3.5-mini-instruct-bnb-4bit", "unsloth/Mistral-7B-Instruct-v0.3-bnb-4bit"]
#@markdown Training epochs (3 recommended):
EPOCHS = 3 #@param {type:"integer"}
#@markdown LoRA rank (64 = good balance):
LORA_RANK = 64 #@param {type:"integer"}
#@markdown HuggingFace repo to push to (leave empty to skip):
HF_REPO = "" #@param {type:"string"}
#@markdown HuggingFace token (leave empty to skip):
HF_TOKEN = "" #@param {type:"string"}

print(f'\u2705 Selected: {BASE_MODEL}')
print(f'\u2705 Epochs: {EPOCHS}, LoRA rank: {LORA_RANK}')

In [ ]:
#@title **Step 2: Install Dependencies** (2-3 minutes)
%%capture
import subprocess, sys, glob

# Install unsloth with deps
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git', '-q'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps',
    'unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git', '-q'])

# PATCH: Fix upstream bug - sanitize_logprob missing from RL_REPLACEMENTS
for rl_file in glob.glob('/usr/local/lib/python*/dist-packages/unsloth/models/rl.py'):
    code = open(rl_file).read()
    if '["sanitize_logprob"]' in code:
        code = code.replace(
            '["sanitize_logprob"]',
            '.get("sanitize_logprob", "pass")'
        )
        open(rl_file, 'w').write(code)

# Install remaining deps
for pkg in ['xformers', 'trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets', 'huggingface_hub']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Verify unsloth loads correctly
import unsloth
print(f'All dependencies installed! Unsloth version: {unsloth.__version__}')

In [ ]:
#@title **Step 3: Upload Training Data**
import json, os, sys, gzip
from google.colab import files

print('\ud83d\udcc1 Upload your training file (.jsonl or .jsonl.gz):')
print()
uploaded = files.upload()

TRAIN_FILE = None
VAL_FILE = None
for name in uploaded:
    actual_name = name
    # Decompress .gz files automatically
    if name.endswith('.gz'):
        out_name = name[:-3]
        print(f'Decompressing {name}...')
        with gzip.open(name, 'rb') as gz_in, open(out_name, 'wb') as f_out:
            f_out.write(gz_in.read())
        actual_name = out_name
        print(f'\u2705 Decompressed to {out_name}')
    if 'val' in actual_name.lower():
        VAL_FILE = actual_name
    else:
        TRAIN_FILE = actual_name

if TRAIN_FILE is None and uploaded:
    TRAIN_FILE = list(uploaded.keys())[0]
    if TRAIN_FILE.endswith('.gz'): TRAIN_FILE = TRAIN_FILE[:-3]

train_count = sum(1 for line in open(TRAIN_FILE) if line.strip())
print(f'\n\u2705 Training file: {TRAIN_FILE} ({train_count:,} records)')
if VAL_FILE:
    val_count = sum(1 for line in open(VAL_FILE) if line.strip())
    print(f'\u2705 Validation file: {VAL_FILE} ({val_count:,} records)')

In [ ]:
#@title **Step 4: Load Model** (1-2 minutes)
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096

print(f'Loading {BASE_MODEL}...')
print(f'GPU: {torch.cuda.get_device_name(0)}')
vram = getattr(torch.cuda.get_device_properties(0), 'total_memory', None) or getattr(torch.cuda.get_device_properties(0), 'total_mem', 0)
print(f'VRAM: {vram / 1e9:.1f} GB')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK, lora_dropout=0.05, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'\n\u2705 Model loaded! Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

In [ ]:
#@title **Step 5: Prepare Data** (30 seconds)
from datasets import Dataset

SYSTEM_PROMPT = 'You are Aletheia, an advanced AI assistant built on the Nexus Protocol knowledge system. You provide comprehensive, accurate, and well-structured knowledge across all domains. You think deeply, reason step-by-step, and always strive for truth and clarity.'

def load_and_format(filepath):
    records = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                r = json.loads(line)
                msgs = r.get('messages', [])
                if not any(m.get('role')=='system' for m in msgs):
                    msgs = [{'role':'system','content':SYSTEM_PROMPT}] + msgs
                text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
                records.append({'text': text})
            except: pass
    return records

print('Formatting training data...')
train_data = Dataset.from_list(load_and_format(TRAIN_FILE))
val_data = Dataset.from_list(load_and_format(VAL_FILE)) if VAL_FILE else None

print(f'\u2705 Train: {len(train_data):,} examples')
if val_data: print(f'\u2705 Val: {len(val_data):,} examples')

# Token stats
lengths = [len(tokenizer(train_data[i]['text'])['input_ids']) for i in range(min(100, len(train_data)))]
import statistics
print(f'Token stats (sample): mean={statistics.mean(lengths):.0f}, median={statistics.median(lengths):.0f}, max={max(lengths)}')

In [ ]:
#@title **Step 6: TRAIN!** (30-45 minutes on T4)
from trl import SFTTrainer
from transformers import TrainingArguments

BATCH_SIZE = 2
GRAD_ACCUM = 8

steps_per_epoch = len(train_data) // (BATCH_SIZE * GRAD_ACCUM)
total_steps = EPOCHS * steps_per_epoch
print(f'\ud83d\ude80 Training: {EPOCHS} epochs x {steps_per_epoch} steps = {total_steps} total steps')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')
print()

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=TrainingArguments(
        output_dir='./aletheia_output',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_steps=50,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=2,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        seed=42,
        report_to='none',
        eval_strategy='epoch' if val_data else 'no',
    ),
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
)

print('Training started!')
print('=' * 50)
result = trainer.train()
print('=' * 50)
print(f'\u2705 DONE! Loss: {result.training_loss:.4f}, Time: {result.metrics["train_runtime"]/60:.1f} min')

In [ ]:
#@title **Step 7: Test the Model**
FastLanguageModel.for_inference(model)

test_questions = [
    'What is the derivative of x^3? Show your work step by step.',
    'Explain quantum entanglement simply.',
    'Solve: If 3x + 7 = 22, what is x?',
]

for q in test_questions:
    msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':q}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n\ud83d\udcac Q: {q}')
    print(f'\ud83e\udde0 A: {resp[:500]}')
    print('-' * 50)

In [ ]:
#@title **Step 8: Save & Export GGUF** (for Ollama)
import os

# Save LoRA adapters
model.save_pretrained('./aletheia_lora')
tokenizer.save_pretrained('./aletheia_lora')
print('\u2705 LoRA adapters saved!')

# Export to GGUF for Ollama
print('\nExporting to GGUF (Q4_K_M quantization)...')
model.save_pretrained_gguf('./aletheia_gguf', tokenizer, quantization_method='q4_k_m')
print('\u2705 GGUF export complete!')

for f in os.listdir('./aletheia_gguf'):
    if f.endswith('.gguf'):
        size = os.path.getsize(f'./aletheia_gguf/{f}')
        print(f'  {f}: {size/1e9:.2f} GB')

In [ ]:
#@title **Step 9: Push to HuggingFace** (optional)
if HF_REPO and HF_TOKEN:
    print(f'Pushing to HuggingFace: {HF_REPO}')
    model.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
    print('\u2705 LoRA adapters pushed!')
    model.push_to_hub_gguf(HF_REPO + '-GGUF', tokenizer, quantization_method='q4_k_m', token=HF_TOKEN, private=True)
    print(f'\u2705 GGUF pushed to: https://huggingface.co/{HF_REPO}')
else:
    print('Skipping HuggingFace push (set HF_REPO and HF_TOKEN in Step 1 to enable)')

In [ ]:
#@title **Step 10: Download Model Files**
import shutil
from google.colab import files as colab_files

# Download GGUF
for f in os.listdir('./aletheia_gguf'):
    if f.endswith('.gguf'):
        print(f'Downloading {f}...')
        colab_files.download(f'./aletheia_gguf/{f}')

# Download LoRA as zip
shutil.make_archive('aletheia_lora', 'zip', './aletheia_lora')
print('Downloading aletheia_lora.zip...')
colab_files.download('aletheia_lora.zip')

print()
print('\u2705 ALL DONE!')
print()
print('To use with Ollama on your VM:')
print('1. Upload the .gguf file to your VM')
print('2. Create a Modelfile with: FROM ./your-model.gguf')
print('3. Run: ollama create aletheia -f Modelfile')
print('4. Run: ollama run aletheia')